# SpaceRL — Versión 3: SpaceMissionEnv + Lógica Difusa

> **Proyecto:** SpaceRL  
> **Versión:** 3 — Entorno personalizado + Sistema de Lógica Difusa  
> **Algoritmo:** Q-learning tabular implementado manualmente  
> **Autores:** Álvaro · Marta · Paula

---

## ¿Qué es la Versión 3?

| Elemento | V1 | V2 | V3 |
|---|---|---|---|
| Entorno | Taxi-v3 estándar | Taxi-v3 + wrappers | SpaceMissionEnv (desde cero) |
| Estados | 500 (opacos) | 500 (recodificados) | 972 (semánticos nativos) |
| Acciones | 6 (movimiento) | 6 (safety-lock) | 6 (estratégicas) |
| Recompensa | Nativa Taxi | Shaped via wrapper | Diseñada desde cero |
| Eventos | No | No | Tormenta, asteroide, zona segura |
| **Lógica difusa** | **No** | **No** | **Sí — Mission Health Index** |

---

## 0. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import (
    N_STATES, N_ACTIONS, N_EPISODES, MAX_STEPS, MISSION_SUCCESS_STEPS,
    LEARNING_RATE, DISCOUNT_FACTOR, EPSILON_START, EPSILON_DECAY,
    N_EVAL_EPS, QTABLE_PATH, METRICS_CSV_PATH, EVAL_CSV_PATH,
    FIGURE_PATH, V1_METRICS_CSV, V2_METRICS_CSV, ACTION_NAMES,
)
from agent import QLearningAgent
from environment import SpaceMissionEnv
from training import run_training_v3
from evaluation import evaluate_agent_v3, build_training_figure, build_three_way_figure
from utils import (
    save_training_metrics_v3, load_training_metrics_v3,
    save_evaluation_metrics_v3, load_evaluation_metrics_v3,
    training_summary_v3, evaluation_summary_v3,
    load_version_metrics, build_three_way_comparison, three_way_summary,
)
from fuzzy import (
    evaluate_mission_state, get_memberships,
    decode_all_states, qtable_fuzzy_analysis, save_fuzzy_analysis,
    plot_membership_functions, plot_mhi_distribution,
    plot_mhi_vs_qvalue, plot_fuzzy_rules_example,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('OK')

## 1. Hiperparámetros

In [ ]:
pd.DataFrame([
    {'Parámetro': 'N_STATES',             'Valor': N_STATES,             'Categoría': 'Entorno'},
    {'Parámetro': 'N_ACTIONS',            'Valor': N_ACTIONS,            'Categoría': 'Entorno'},
    {'Parámetro': 'MAX_STEPS',            'Valor': MAX_STEPS,            'Categoría': 'Entorno'},
    {'Parámetro': 'MISSION_SUCCESS_STEPS','Valor': MISSION_SUCCESS_STEPS,'Categoría': 'Entorno'},
    {'Parámetro': 'N_EPISODES',           'Valor': N_EPISODES,           'Categoría': 'Training'},
    {'Parámetro': 'LEARNING_RATE',        'Valor': LEARNING_RATE,        'Categoría': 'Training'},
    {'Parámetro': 'DISCOUNT_FACTOR',      'Valor': DISCOUNT_FACTOR,      'Categoría': 'Training'},
    {'Parámetro': 'EPSILON_START',        'Valor': EPSILON_START,        'Categoría': 'Training'},
    {'Parámetro': 'EPSILON_DECAY',        'Valor': EPSILON_DECAY,        'Categoría': 'Training'},
]).style.set_caption('Configuración V3')

## 2. Exploración del entorno

In [ ]:
env = SpaceMissionEnv()
obs, _ = env.reset(seed=42)
print(f'States: {env.observation_space.n}  |  Actions: {env.action_space.n}')
print(f'Initial state: {obs}  →  {SpaceMissionEnv.decode_state(obs)}')
env.close()

In [ ]:
rows = []
for s in [0, 100, 200, 400, 600, 800, 971]:
    d = SpaceMissionEnv.decode_state(s)
    d['state_id'] = s
    rows.append(d)
pd.DataFrame(rows)[['state_id','energy','oxygen','food','fuel','hull','event']].style.set_caption('Decodificación semántica')

## 3. Entrenamiento
> ⚠️ Salta esta celda si ya tienes `results/metrics/metrics_v3.csv`

In [ ]:
env   = SpaceMissionEnv()
agent = QLearningAgent(env)
print(f'Entrenando {N_EPISODES:,} episodios...\n')
metrics = run_training_v3(env, agent)
env.close()
agent.save(QTABLE_PATH)
df_train = save_training_metrics_v3(metrics, csv_path=METRICS_CSV_PATH)
print('\nEntrenamiento completado.')

## 4. Análisis de métricas

In [ ]:
df_train = load_training_metrics_v3(METRICS_CSV_PATH)
df_train.head(8)

In [ ]:
df_train[['reward','steps','td_error','epsilon','success']].describe().round(3)

In [ ]:
summary = training_summary_v3(df_train, last_n=1000)
display(summary.style.set_caption('Resumen V3 — últimos 1000 episodios'))

In [ ]:
if 'term_reason' in df_train.columns:
    display(df_train['term_reason'].value_counts().rename('Episodios').to_frame().style.set_caption('Razones de terminación'))

## 5. Visualización

In [ ]:
build_training_figure(df_train, save_path=FIGURE_PATH, show=True)

## 6. Evaluación

In [ ]:
agent_eval = QLearningAgent()
agent_eval.load(QTABLE_PATH)
results = evaluate_agent_v3(agent_eval, n_episodes=N_EVAL_EPS)
df_eval = save_evaluation_metrics_v3(
    results['rewards'], results['steps'], results['successes'],
    csv_path=EVAL_CSV_PATH,
)
display(evaluation_summary_v3(df_eval).style.set_caption('Resumen evaluación V3'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('SpaceRL V3 — Distribuciones de evaluación', fontsize=12, fontweight='bold')
for ax, col, label, color in [
    (axes[0], 'reward',  'Recompensa', '#2196F3'),
    (axes[1], 'steps',   'Pasos',      '#4CAF50'),
    (axes[2], 'success', 'Éxito',      '#FF9800'),
]:
    ax.hist(df_eval[col], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df_eval[col].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Media: {df_eval[col].mean():.2f}')
    ax.set_title(label); ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()

## 7. Política aprendida

In [ ]:
qtable = agent_eval.qtable
print(f'Q-table: {qtable.shape} | Rango: [{qtable.min():.2f}, {qtable.max():.2f}]')
rows = []
for s in [0, 100, 200, 400, 600, 800, 971]:
    desc = SpaceMissionEnv.decode_state(s)
    row = {'Estado': s, 'Energía': desc['energy'], 'Evento': desc['event'],
           'Acción óptima': ACTION_NAMES[np.argmax(qtable[s])]}
    for a, name in enumerate(ACTION_NAMES):
        row[name] = round(qtable[s, a], 2)
    rows.append(row)
pd.DataFrame(rows).style.set_caption('Política aprendida').highlight_max(subset=ACTION_NAMES, axis=1, color='#c8e6c9')

In [ ]:
best_actions = np.argmax(qtable, axis=1)
action_counts = pd.Series(best_actions).value_counts().sort_index()
action_counts.index = [ACTION_NAMES[i] for i in action_counts.index]
fig, ax = plt.subplots(figsize=(10, 4))
action_counts.plot(kind='bar', ax=ax, color=['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#00BCD4'], edgecolor='white', alpha=0.85)
ax.set_title('Acción greedy preferida por estado', fontsize=11, fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

---
## 8. Lógica Difusa — Mission Health Index

### Fundamento teórico (Tema 5: Razonamiento con Incertidumbre)

| Conjunto | Función | Descripción |
|---|---|---|
| LOW | trimf(0, 0, 1) | Recurso crítico |
| MEDIUM | trimf(0, 1, 2) | Recurso estable |
| HIGH | trimf(1, 2, 2) | Recurso óptimo |

### Reglas difusas IF-THEN

| Regla | Antecedente | Consecuente |
|---|---|---|
| R1 | IF energy is LOW | RISK is CRITICAL |
| R2 | IF oxygen is LOW | RISK is CRITICAL |
| R3 | IF hull is LOW | RISK is CRITICAL |
| R4 | IF food is LOW OR fuel is LOW | RISK is HIGH |
| R5 | IF all resources are MEDIUM | RISK is MODERATE |
| R6 | IF all resources are HIGH | RISK is SAFE |
| R7 | IF energy AND oxygen AND hull are HIGH | RISK is SAFE |

**Defuzzificación:** método del centroide  
**MHI** = 1 − (crisp_risk / 3) ∈ [0, 1]

In [ ]:
plot_membership_functions(save_path='results/figures/fuzzy_membership_functions.png')

In [ ]:
examples = [
    (0, 0, 0, 0, 0, 'All CRITICAL'),
    (1, 1, 1, 1, 1, 'All STABLE'),
    (2, 2, 2, 2, 2, 'All OPTIMAL'),
    (0, 2, 1, 1, 2, 'Energy LOW, rest OK'),
    (1, 0, 1, 1, 1, 'Oxygen LOW'),
    (2, 2, 0, 1, 2, 'Food LOW'),
]
rows = []
for e, o, f, u, h, label in examples:
    r = evaluate_mission_state(e, o, f, u, h)
    rows.append({'Escenario': label, 'Energy': e, 'Oxygen': o, 'Food': f,
                 'Fuel': u, 'Hull': h, 'Crisp Risk': r['crisp_risk'],
                 'MHI': r['mhi'], 'Status': r['status']})
pd.DataFrame(rows).style.set_caption('Evaluación difusa').background_gradient(subset=['MHI'], cmap='RdYlGn')

In [ ]:
plot_fuzzy_rules_example(save_path='results/figures/fuzzy_rules_example.png')

In [ ]:
df_fuzzy = qtable_fuzzy_analysis(agent_eval.qtable)
save_fuzzy_analysis(df_fuzzy, path='results/comparisons/fuzzy_analysis.csv')
df_fuzzy[['state_id','energy','hull','event','mhi','status','greedy_action','greedy_qvalue']].head(10)

In [ ]:
status_summary = df_fuzzy.groupby('status').agg(
    n_states=('state_id','count'),
    mean_mhi=('mhi','mean'),
    mean_qvalue=('greedy_qvalue','mean'),
    pct_appropriate=('appropriate_action','mean')
).round(3)
status_summary.columns = ['Nº estados','MHI medio','Q-value medio','% acciones apropiadas']
display(status_summary.style.set_caption('Resumen por estado difuso').background_gradient(subset=['MHI medio'], cmap='RdYlGn'))

In [ ]:
plot_mhi_distribution(df_fuzzy, save_path='results/figures/fuzzy_mhi_distribution.png')

In [ ]:
plot_mhi_vs_qvalue(df_fuzzy, save_path='results/figures/fuzzy_mhi_vs_qvalue.png')

In [ ]:
corr = df_fuzzy['mhi'].corr(df_fuzzy['greedy_qvalue'])
print(f'Correlación MHI vs Q-value: r = {corr:.4f}')
if corr > 0.5:
    print('Correlación positiva fuerte: el agente valora más los estados saludables.')
elif corr > 0.2:
    print('Correlación positiva moderada.')
else:
    print('Correlación baja: el agente necesita más entrenamiento.')

## 9. Comparación V1 vs V2 vs V3

In [ ]:
df_v1 = load_version_metrics(V1_METRICS_CSV, 'V1')
df_v2 = load_version_metrics(V2_METRICS_CSV, 'V2')
display(three_way_summary(df_v1, df_v2, df_train, last_n=1000).style.set_caption('V1 vs V2 vs V3'))

In [ ]:
cmp_fig = FIGURE_PATH.replace('training_metrics_v3', 'v1_v2_v3_comparison')
build_three_way_figure(df_v1, df_v2, df_train, save_path=cmp_fig, show=True)

## 10. Resumen final

In [ ]:
s_train = training_summary_v3(df_train, last_n=1000)
s_eval  = evaluation_summary_v3(df_eval)
print('=' * 65)
print('  SpaceRL V3 — SpaceMissionEnv + Lógica Difusa  |  Resultados')
print('=' * 65)
print(f"  Recompensa media  : {s_train['mean_reward'].values[0]:.2f}")
print(f"  Pasos medios      : {s_train['mean_steps'].values[0]:.1f}")
print(f"  Tasa de éxito     : {s_train['success_rate_%'].values[0]:.1f}%")
print(f"  Evaluación éxito  : {s_eval['success_rate_%'].values[0]:.1f}%")
print()
print('  Lógica difusa:')
print(f"  MHI medio         : {df_fuzzy['mhi'].mean():.4f}")
print(f"  Estados OPTIMAL   : {(df_fuzzy['status']=='OPTIMAL').sum()}")
print(f"  Estados CRITICAL  : {(df_fuzzy['status']=='CRITICAL').sum()}")
print(f"  Corr MHI/Q-value  : {df_fuzzy['mhi'].corr(df_fuzzy['greedy_qvalue']):.4f}")
print('=' * 65)